# Evaluacion de rendimiento

Usamos un enfoque de **Pipeline + GridSearchCV** porque este dataset tiene alrededor de **700 instancias**, un tamaño intermedio en el que podemos permitirnos validación cruzada sin que el coste computacional sea excesivo.

**Por qué Pipeline:**
- Encadena en un solo objeto todos los pasos de preprocesado y modelado (detección de outliers, escalado y clasificador).
- Garantiza que cada transformación se ajusta solo con los datos de entrenamiento de cada fold, evitando fuga de información.
- Hace el flujo reproducible y más fácil de mantener.

**Por qué GridSearchCV:**
- Permite buscar sistemáticamente la mejor combinación de hiperparámetros del KNN (`n_neighbors`, `weights`, `p`).
- En un conjunto de ~700 filas, una búsqueda en malla con validación cruzada de 5 folds sigue siendo viable.
- Devuelve el mejor modelo ya reentrenado y métricas por combinación para comparar decisiones.

**Por qué StratifiedKFold:**
- En este problema de clasificación, la variable objetivo puede estar desbalanceada.
- `StratifiedKFold` mantiene la proporción de clases en cada fold, reduciendo varianza en la estimación y haciendo la evaluación más estable que un KFold simple.

In [ ]:
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn import preprocessing

pipeline = Pipeline([('outlier_detector', OutlierDetection(k=3)),
                     ('standardization', preprocessing.StandardScaler()),
                     ('model', KNeighborsClassifier())])

params_grid = {
    #Habria que añadir el mejor k para el Outlier
    "model__n_neighbors": [3, 5, 7, 9, 11, 13, 15],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 1.5, 2, 3]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = GridSearchCV(estimator=pipeline,
                      n_jobs=-2, #Cogemos los nucleos para un entrenamiento mas rapido
                      param_grid=params_grid,
                      scoring='neg_root_mean_squared_error',  # por ahora cogemos esta
                      cv=cv,
                      return_train_score=True)

models.fit(X_aux, y_aux)

#Predecimos con el mejor modelo
best_model = models.best_estimator_
pred_test = best_model.predict(X_test)
RMSE_test = root_mean_squared_error(y_test, pred_test)


#Comprobamos el rendimiento en Validation y Test
print('Rendimiento en Validacion cruzada : \n', -models.best_score_)
print('Rendimiento en Test : ',RMSE_test)



# Analisis de resultados

Holi